# Image Cluster Notebook
This notebook is designed to create labels for NAC images through K-Means clustering. It does 2 main passes: the initial K-means pass where the algorithm infers pixel grouping from their pixel values. A second pass is then done where the user must select the cluster values that belong to class 0 (no crater) and 1 (crater). These are then submitted to the display function and shown at the end of the notebook. 

## Imports

In [1]:
# PROJ must be configured before rasterio/localtileserver are imported.
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

from pathlib import Path
import sys
import ipysheet
from IPython.display import Markdown, display
import ipywidgets
import leafmap
import numpy
import pandas
import rasterio
from rasterio.windows import Window
from localtileserver import TileClient, get_leaflet_tile_layer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from ipyleaflet import WidgetControl

repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from model.clustering.Clusterer import Clusterer
from model.clustering.ImageHelperSingleBand import ImageHelper
from model.clustering.clustering_display_utils import display_images_labels, display_images_binary_labels
from model.clustering.clustering_backend_utils import crop_center, handleClick, relabel

# Configuration

`inFile`: Input single-band NAC file to perform clustering on.

`noDataValue`: Nodata value to ignore in clustering. This is vital for the clustering algorithm to properly capture the valid data distribution.

`numClusters`: Number of K-means clusters to create. Higher values result in a more noisy output (making it harder to discern between cluster groups by eye), while lower values create more homogenous clusters (which makes for poor labels).

`cropSize`: Target size for a centered square crop used for clustering and display.

In [2]:
# Original full-resolution lunar raster.
inFile = "/explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF"

# Outputs are written here.
outDirectory = ""

# This raster uses 0 as the fill/nodata value around the valid strip.
noDataValue = 0.0

# Number of K-Means clusters to use.
numClusters = 20

# Centered crop size used for this debugging workflow.
cropSize = 5000

## Path setup

In [3]:
inFile = Path(inFile)

outDirectory = Path(outDirectory)
outDirectory.mkdir(parents=True, exist_ok=True)

clippedInputFile = outDirectory / f"{inFile.stem}-clip-{cropSize}{inFile.suffix}"
labelsFile = outDirectory / f"{inFile.stem}-clip-{cropSize}-labels{inFile.suffix}"
clusterMapFile = outDirectory / f"{inFile.stem}-clip-{cropSize}-cluster-map{inFile.suffix}"

# Step 1: Clip the input raster

In [4]:
crop_center(
    src_path=inFile,
    dst_path=clippedInputFile,
    size=cropSize,
)

print(f"Raw input:       {inFile}")
print(f"Clipped input:   {clippedInputFile}")
print(f"Cluster labels:  {labelsFile}")
print(f"Final label map: {clusterMapFile}")

Raw input:       /explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF
Clipped input:   NAC_DTM_NEWCRATER6_M1219245090_80CM-clip-5000.TIF
Cluster labels:  NAC_DTM_NEWCRATER6_M1219245090_80CM-clip-5000-labels.TIF
Final label map: NAC_DTM_NEWCRATER6_M1219245090_80CM-clip-5000-cluster-map.TIF


# Step 2: Ingest the clipped raster

In [5]:
inHelper = ImageHelper()
inHelper.initFromFile(
    inputFile=clippedInputFile,
    noDataValue=noDataValue,
)

print(f"Clustering input shape: {inHelper.getBand().shape}")

Clustering input shape: (5000, 5000)


# Step 3: Generate first-pass clusters on the clipped raster

These clusters will have many different groupings, equal to numClusters. The second pass will narrow them down to only 2 classes (no crater, crater).

In [6]:
# Add singleton dimension for the single input band: (H, W) -> (1, H, W).
labels = Clusterer.getClusters(
    bands=numpy.expand_dims(inHelper.getBand(), axis=0),
    numClusters=numClusters,
    noDataValue=noDataValue,
)

# Because inHelper was created from clippedInputFile, the label GeoTIFF is
# written with the same clipped extent/transform/CRS.
labelsDs = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    labelsFile,
    labels,
)

lHelper = ImageHelper()
lHelper.initFromDataset(labelsDs, noDataValue)

print(f"Clipped labels: {labelsFile}")

K-Means predict: 100%|██████████| 96/96 [00:00<00:00, 2073.94batch/s]


Clipped labels: NAC_DTM_NEWCRATER6_M1219245090_80CM-clip-5000-labels.TIF


In [11]:
band = inHelper.getBand()
valid = band != noDataValue

print("shape:", band.shape)
print("valid fraction:", valid.mean())
print("valid pixels:", valid.sum())
print("unique raw values:", numpy.unique(band[valid]).size)

print("valid percentiles:")
for p in [0, 0.1, 1, 5, 25, 50, 75, 95, 99, 99.9, 100]:
  print(p, numpy.percentile(band[valid], p))

unique, counts = numpy.unique(labels, return_counts=True)
for label, count in zip(unique, counts):
  print(label, count, count / labels.size)

shape: (5000, 5000)
valid fraction: 1.0
valid pixels: 25000000
unique raw values: 2655
valid percentiles:
0 -0.00013848285
0.1 0.0078012003
1 0.011647946
5 0.013125096
25 0.014463764
50 0.015340822
75 0.01632559
95 0.018341282
99 0.0208032
99.9 0.02569626
100 0.06022465
1 7653226 0.30612904
7 2333895 0.0933558
8 227964 0.00911856
11 999230 0.0399692
12 100160 0.0040064
15 2606749 0.10426996
16 6297531 0.25190124
17 4267887 0.17071548
18 513358 0.02053432


# Step 4: Display clipped image + clipped labels

In [7]:
m, legend_control = display_images_labels(clippedInputFile, labelsFile, labels, inHelper, lHelper)
display(m)

INFO:     Started server process [674900]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:44487 (Press CTRL+C to quit)


INFO:     127.0.0.1:53244 - "GET /api/metadata?&filename=%2Fpanfs%2Fccds02%2Fnobackup%2Fpeople%2Fajkerr1%2FLunar_FM%2Ffull_model_lfm%2Flfm%2Fnotebooks%2FNAC_DTM_NEWCRATER6_M1219245090_80CM-clip-5000.TIF HTTP/1.1" 200 OK


INFO:     127.0.0.1:53250 - "GET /api/metadata?&filename=%2Fpanfs%2Fccds02%2Fnobackup%2Fpeople%2Fajkerr1%2FLunar_FM%2Ffull_model_lfm%2Flfm%2Fnotebooks%2FNAC_DTM_NEWCRATER6_M1219245090_80CM-clip-5000-labels.TIF HTTP/1.1" 200 OK


Map(center=[-6.431664, -44.048283], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Update the labels
Select multiple values by clicking the mouse or using the arrow keys while pressing Shift, Control, or Command.

In [10]:
opts = list(numpy.unique(labels))

sl = ipywidgets.SelectMultiple(
    options=opts,
    layout=ipywidgets.Layout(height="200px", width="150px"),
)

bt = ipywidgets.ToggleButtons(
    options=["Select:", "Next", "Done", "Start Over"],
    value="Select:",
)

output = ipywidgets.Output()
display(sl, bt, output)
table = {}
bt.observe(lambda change: handleClick(change, output, sl, bt, opts, table), names="value")

SelectMultiple(layout=Layout(height='200px', width='150px'), options=(np.int32(1), np.int32(7), np.int32(8), n…

ToggleButtons(options=('Select:', 'Next', 'Done', 'Start Over'), value='Select:')

Output()

## Edit the groups
Edit cluster IDs in each group. When finished, proceed to the next cell.

In [9]:
strTab = {}

for item in table:
    strTab[item] = ", ".join(str(i) for i in table[item])

df = pandas.DataFrame(strTab.items(), columns=["Class", "Cluster ID"])
sheet = ipysheet.from_dataframe(df)
sheet.column_width = [1, 5]
sheet

IndexError: list index out of range

In [ ]:
editedDf = ipysheet.to_dataframe(sheet)
strClusters = editedDf.to_dict()["Cluster ID"]

finalClusters = {}

for key in strClusters:
    strCluster = strClusters[key]
    finalClusters[int(key)] = [
        int(i.strip()) for i in strCluster.split(",") if i.strip()
    ]

print(finalClusters)
newClusters = relabel(labels, finalClusters)

## Review the updated map

In [ ]:
m = display_images_binary_labels(m, inHelper, clusterMapFile, labelsFile, newClusters)
display(m)